# Assertion test cho VAE và checkpoint
Chọn **Run All**. Cell thiết lập tìm mã nguồn trong thư mục hiện tại hoặc thư mục cha. Trên Colab, cell dùng bản có sẵn ở `/content/DL_ImgGenModel` hoặc tải nhánh `khanh` nếu thư mục này chưa tồn tại. Để test mã nguồn riêng ở vị trí khác, đặt `PROJECT_ROOT` trong cell thiết lập. Các test dùng tensor tổng hợp và thư mục tạm cho checkpoint.

In [ ]:
import math
import subprocess
import sys
import tempfile
import unittest
from pathlib import Path
from unittest.mock import patch

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = None  # Có thể đặt thành đường dẫn tới thư mục chứa models/ và utils/.
colab_repo = Path('/content/DL_ImgGenModel')
if PROJECT_ROOT is None:
    cwd = Path.cwd()
    candidates = [cwd, *cwd.parents, cwd / 'DL_ImgGenModel']
    if 'google.colab' in sys.modules or Path('/content').is_dir():
        candidates.append(colab_repo)
    PROJECT_ROOT = next((p for p in candidates if (p / 'models/vae/train.py').is_file()), None)
    if PROJECT_ROOT is None and colab_repo in candidates and not colab_repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'khanh',
                        'https://github.com/lucaspham102/DL_ImgGenModel.git', str(colab_repo)], check=True)
        PROJECT_ROOT = colab_repo
if PROJECT_ROOT is None or not (Path(PROJECT_ROOT) / 'models/vae/train.py').is_file():
    raise FileNotFoundError('Hãy đặt PROJECT_ROOT thành thư mục dự án chứa models/vae/train.py rồi chạy lại cell.')
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Mã nguồn đang dùng: {PROJECT_ROOT}')

from models.vae import VAE, train
from utils.logger import load_checkpoint, save_checkpoint


def run_group(test_class):
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(test_class)
    return unittest.TextTestRunner(stream=sys.stdout, verbosity=2, buffer=True).run(suite)

Mã nguồn đang dùng: /content/DL_ImgGenModel


## Model VAE

In [ ]:
class VAEModelTest(unittest.TestCase):
    def setUp(self):
        self.addCleanup(torch.set_rng_state, torch.get_rng_state())
        torch.default_generator.manual_seed(42)

    def assert_images(self, images, shape):
        self.assertEqual(tuple(images.shape), shape)
        self.assertTrue(torch.isfinite(images).all().item())
        self.assertGreaterEqual(images.min().item(), -1.0)
        self.assertLessEqual(images.max().item(), 1.0)

    @torch.no_grad()
    def test_forward_matches_encoder_and_decoder(self):
        for model, channels, latent, batch in ((VAE(), 3, 128, 2), (VAE(1, 5), 1, 5, 1)):
            with self.subTest(channels=channels, batch=batch):
                model.eval()
                images = torch.rand(batch, channels, 64, 64) * 2 - 1
                expected_mu, expected_logvar = model.encode(images)
                rng = torch.get_rng_state()
                expected_images = model.decode(model.reparameterize(expected_mu, expected_logvar))
                torch.set_rng_state(rng)
                recon, mu, logvar = model(images)
                self.assert_images(recon, tuple(images.shape))
                for actual, expected in ((mu, expected_mu), (logvar, expected_logvar)):
                    self.assertEqual(tuple(actual.shape), (batch, latent))
                    torch.testing.assert_close(actual, expected)
                torch.testing.assert_close(recon, expected_images)

    def test_reparameterization_and_gradients(self):
        model = VAE(latent_dim=2)
        mu = torch.tensor([[2., -1.]], requires_grad=True)
        logvar = torch.tensor([[math.log(4.), math.log(0.25)]], requires_grad=True)
        rng = torch.get_rng_state()
        epsilon = torch.randn_like(mu)
        torch.set_rng_state(rng)
        z = model.reparameterize(mu, logvar)
        perturbation = torch.tensor([[2., 0.5]]) * epsilon
        torch.testing.assert_close(z, mu + perturbation)
        z.sum().backward()
        torch.testing.assert_close(mu.grad, torch.ones_like(mu))
        torch.testing.assert_close(logvar.grad, perturbation / 2)

    def test_loss_matches_hand_calculation(self):
        # Hai ảnh có SSE bằng 5 và 1. Phương sai latent lần lượt là 4 và 1.
        recon = torch.tensor([[[[1., 2.]]], [[[0., 1.]]]])
        images = torch.zeros_like(recon)
        mu = torch.tensor([[1.], [0.]])
        logvar = torch.tensor([[math.log(4.)], [0.]])
        expected_kl = (4 - math.log(4)) / 4
        for beta in (0., 0.5, 1.):
            with self.subTest(beta=beta):
                losses = VAE.loss_function(recon, images, mu, logvar, kl_weight=beta)
                expected = {"recon_loss": 3., "recon_mse": 1.5, "kl_loss": expected_kl,
                            "loss": 3 + beta * expected_kl}
                for name, value in expected.items():
                    torch.testing.assert_close(losses[name], torch.tensor(value), msg=name)

    def test_extreme_log_variance_stays_finite(self):
        model = VAE(latent_dim=2).eval()
        images = torch.zeros(1, 3, 64, 64)
        raw = torch.tensor([[-1000., 1000.]])
        with torch.no_grad():
            model.encoder.fc_logvar.weight.zero_()
            model.encoder.fc_logvar.bias.copy_(raw[0])
            mu, logvar = model.encode(images)
        bounds = torch.tensor([[model.LOGVAR_MIN, model.LOGVAR_MAX]])
        torch.testing.assert_close(logvar, bounds)
        self.assertTrue(torch.isfinite(model.reparameterize(mu, raw)).all().item())
        losses = model.loss_function(images, images, mu, raw)
        bounded = model.loss_function(images, images, mu, bounds)
        self.assertTrue(torch.isfinite(losses["loss"]).item())
        torch.testing.assert_close(losses["kl_loss"], bounded["kl_loss"])

    def test_training_step_updates_encoder_and_decoder(self):
        model = VAE(latent_dim=4).train()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        images = torch.rand(2, 3, 64, 64) * 2 - 1
        tracked = [model.encoder.fc_mu.weight, model.encoder.fc_logvar.weight,
                   model.decoder.fc[0].weight]
        before = [p.detach().clone() for p in tracked]
        recon, mu, logvar = model(images)
        loss = model.loss_function(recon, images, mu, logvar)["loss"]
        self.assertTrue(torch.isfinite(loss).item())
        loss.backward()
        for name, param in model.named_parameters():
            self.assertIsNotNone(param.grad, name)
            self.assertTrue(torch.isfinite(param.grad).all().item(), name)
        optimizer.step()
        for old, param in zip(before, tracked):
            self.assertFalse(torch.equal(old, param))

    def test_inference_helpers_preserve_mode_and_batchnorm(self):
        model = VAE(latent_dim=4).eval()
        images = torch.rand(2, 3, 64, 64) * 2 - 1
        with torch.no_grad():
            mu, _ = model.encode(images)
            expected_recon = model.decode(mu)
            rng = torch.get_rng_state()
            expected_samples = model.decode(torch.randn(3, 4))
        buffers = {name: buf.clone() for name, buf in model.named_buffers()}
        for training in (True, False):
            with self.subTest(training=training):
                model.train(training)
                recon = model.reconstruct(images)
                self.assertEqual(model.training, training)
                torch.testing.assert_close(recon, expected_recon)
                torch.set_rng_state(rng)
                samples = model.sample(3)
                self.assertEqual(model.training, training)
                self.assert_images(samples, (3, 3, 64, 64))
                torch.testing.assert_close(samples, expected_samples)
                self.assertFalse(samples.requires_grad)
                self.assertFalse(recon.requires_grad)
                for name, buf in model.named_buffers():
                    torch.testing.assert_close(buf, buffers[name], msg=name)

    def test_invalid_configuration_and_input(self):
        for kwargs in ({"latent_dim": 0}, {"in_channels": 0}):
            with self.subTest(kwargs=kwargs), self.assertRaises(ValueError):
                VAE(**kwargs)
        model = VAE()
        for shape in ((3, 64, 64), (1, 1, 64, 64), (1, 3, 32, 64)):
            with self.subTest(shape=shape), self.assertRaises(ValueError):
                model(torch.zeros(shape))

In [ ]:
model_result = run_group(VAEModelTest)

test_extreme_log_variance_stays_finite (__main__.VAEModelTest.test_extreme_log_variance_stays_finite) ... ok
test_forward_matches_encoder_and_decoder (__main__.VAEModelTest.test_forward_matches_encoder_and_decoder) ... ok
test_inference_helpers_preserve_mode_and_batchnorm (__main__.VAEModelTest.test_inference_helpers_preserve_mode_and_batchnorm) ... ok
test_invalid_configuration_and_input (__main__.VAEModelTest.test_invalid_configuration_and_input) ... ok
test_loss_matches_hand_calculation (__main__.VAEModelTest.test_loss_matches_hand_calculation) ... ok
test_reparameterization_and_gradients (__main__.VAEModelTest.test_reparameterization_and_gradients) ... ok
test_training_step_updates_encoder_and_decoder (__main__.VAEModelTest.test_training_step_updates_encoder_and_decoder) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.931s

OK


## Checkpoint VAE và resume

In [ ]:
class VAECheckpointTest(unittest.TestCase):
    def setUp(self):
        directory = tempfile.TemporaryDirectory()
        self.addCleanup(directory.cleanup)
        self.directory = Path(directory.name)
        self.last = self.directory / "vae_fashion_mnist_last.pt"
        self.best = self.directory / "vae_fashion_mnist_best.pt"
        self.images = torch.linspace(-1, 1, 2 * 3 * 64 * 64).reshape(2, 3, 64, 64)
        self.loader = DataLoader(TensorDataset(self.images), batch_size=2)

    def run_epoch(self, epoch, val_loss):
        # Cố định điểm validation để thử chính xác từng nhánh chọn checkpoint.
        # Model, backward, optimizer, scheduler và thao tác lưu/nạp đều chạy thật.
        def validate(model, loader, device, args):
            model.eval()
            self.reference = model.reconstruct(self.images).detach().clone()
            return val_loss, val_loss, 0.0

        argv = ["train.py", "--epochs", str(epoch), "--latent-dim", "4",
                "--batch-size", "2", "--backend", "none", "--ckpt-dir", str(self.directory)]
        if epoch > 1:
            argv.extend(["--resume", str(self.last)])
        with (
            patch.object(sys, "argv", argv),
            patch.object(torch.cuda, "is_available", return_value=False),
            patch.object(train, "get_dataloaders", return_value=(self.loader,) * 3),
            patch.object(train, "evaluate", side_effect=validate) as evaluate,
        ):
            train.main()
        evaluate.assert_called_once()

    def check_checkpoint(self, path, epoch, best_loss, reference):
        model = train.VAE(latent_dim=4).eval()
        info = load_checkpoint(path, models={"vae": model}, restore_rng=False)
        self.assertEqual(info["epoch"], epoch, path.name)
        self.assertEqual(info["step"], epoch, path.name)  # Một batch mỗi epoch.
        self.assertEqual(info["extra"]["best_val_loss"], best_loss, path.name)
        torch.testing.assert_close(model.reconstruct(self.images), reference, rtol=0, atol=0)

    def test_first_epoch_saves_both_checkpoints(self):
        self.run_epoch(1, 10.0)
        for path in (self.best, self.last):
            with self.subTest(checkpoint=path.name):
                self.check_checkpoint(path, 1, 10.0, self.reference)

    def test_resume_preserves_best_when_loss_worsens_or_ties(self):
        self.run_epoch(1, 10.0)
        original = self.reference
        for epoch, loss in ((2, 12.0), (3, 10.0)):
            self.run_epoch(epoch, loss)
            with self.subTest(epoch=epoch):
                self.check_checkpoint(self.best, 1, 10.0, original)
                self.check_checkpoint(self.last, epoch, 10.0, self.reference)

    def test_resume_updates_both_when_loss_improves(self):
        self.run_epoch(1, 10.0)
        self.run_epoch(2, 8.0)
        for path in (self.best, self.last):
            with self.subTest(checkpoint=path.name):
                self.check_checkpoint(path, 2, 8.0, self.reference)

In [ ]:
checkpoint_result = run_group(VAECheckpointTest)

test_first_epoch_saves_both_checkpoints (__main__.VAECheckpointTest.test_first_epoch_saves_both_checkpoints) ... 
  test_first_epoch_saves_both_checkpoints (__main__.VAECheckpointTest.test_first_epoch_saves_both_checkpoints) (checkpoint='vae_fashion_mnist_last.pt') ... FAIL

Stdout:
[*] Thiết bị huấn luyện VAE: cpu
[*] Đang nạp dữ liệu 'fashion_mnist' từ '/content/DL_ImgGenModel/data'...
[+] Dữ liệu sẵn sàng: 1 batches train | 1 batches val

============================== BẮT ĐẦU HUẤN LUYỆN VAE (1 EPOCHS) ==============================
Epoch [1/1] | Batch [1/1] | Loss: 6106.13 (Recon: 6106.05, KL: 0.80, beta: 0.100) | Thời gian: 0.0s
[Epoch 01/01 - 0.0s] Train Loss: 6106.13 (Recon: 6106.05, KL: 0.80) | Val Loss: 10.00 (Recon: 10.00, KL: 0.00)
[*] Đã cập nhật Checkpoint tốt nhất: /tmp/tmp2w2gy80z/vae_fashion_mnist_best.pt (Val Loss: 10.00)

[+] Quá trình hoàn tất.
test_resume_preserves_best_when_loss_worsens_or_ties (__main__.VAECheckpointTest.test_resume_preserves_best_when_loss_worsen

## Smoke test checkpoint dùng chung
Kiểm tra assertion của smoke test bằng cùng một vector nhiễu trước và sau khi nạp lại trọng số của generator.

In [ ]:
def tiny_G(nz=100):  # 100 -> 3x32x32
    return nn.Sequential(
        nn.ConvTranspose2d(nz, 128, 4, 1, 0, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),  # 4x4
        nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True),   # 8x8
        nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(True),    # 16x16
        nn.ConvTranspose2d(32, 3, 4, 2, 1, bias=False), nn.Tanh(),                             # 32x32
    )


class GeneratorCheckpointTest(unittest.TestCase):
    def test_round_trip(self):
        with tempfile.TemporaryDirectory() as directory, torch.random.fork_rng(devices=[]):
            torch.default_generator.manual_seed(42)
            model = tiny_G().eval()
            noise = torch.randn(4, 100, 1, 1)
            path = str(Path(directory) / "smoke.pt")
            with torch.no_grad():
                expected = model(noise)
                save_checkpoint(path, {"G": model}, extra={"fixed_noise": noise})
                restored = tiny_G().eval()
                info = load_checkpoint(path, {"G": restored}, restore_rng=False)
                actual = restored(info["extra"]["fixed_noise"])
            torch.testing.assert_close(actual, expected, atol=1e-8, rtol=1e-5)

In [ ]:
smoke_result = run_group(GeneratorCheckpointTest)

## Kết quả toàn bộ

In [ ]:
results = [model_result, checkpoint_result, smoke_result]
total = sum(result.testsRun for result in results)
failures = sum(len(result.failures) for result in results)
errors = sum(len(result.errors) for result in results)
print(f"Đã chạy {total} test. Số assertion thất bại: {failures}. Số lỗi thực thi: {errors}.")
assert all(result.wasSuccessful() for result in results), "Xem chi tiết ở nhóm test thất bại phía trên."